# Mirage — DistilBERT fine-tuning (Kaggle / Colab, T4 GPU)

Thin notebook: all logic lives in the `mirage` package. Steps:
1. Turn on **GPU T4** and **Internet** in the notebook settings.
2. `REPO_URL` below points at the project repo (change it if you fork).
3. Run all cells. Each seed takes about 20–30 minutes.
4. Download `distilbert_models.zip` from the output panel and unzip it into `artifacts/models/` on your laptop.

In [ ]:
REPO_URL = "https://github.com/AditeyaAItronics/mirage.git"
SEEDS = [13, 42, 87]  # fallback if short on GPU time: [42]
SMOKE_TEST = False  # True = 2,000 examples, ~2 minutes, to check everything works first

In [ ]:
import os, subprocess
os.chdir("/kaggle/working") if os.path.isdir("/kaggle/working") else None
if not os.path.isdir("mirage"):
    subprocess.run(["git", "clone", REPO_URL, "mirage"], check=True)
os.chdir("mirage")
!pip install -q -e ".[dl]"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU!")

In [ ]:
!python -m mirage prepare

In [ ]:
limit = "--limit 2000" if SMOKE_TEST else ""
seeds = " ".join(map(str, SEEDS))
!python -m mirage train-distilbert --seeds {seeds} {limit}

In [ ]:
# Package only the final weights + tokenizer + logs (no intermediate checkpoints).
import shutil, glob
for d in glob.glob("artifacts/models/distilbert__*/checkpoints"):
    shutil.rmtree(d)
shutil.make_archive("/kaggle/working/distilbert_models" if os.path.isdir("/kaggle/working") else "distilbert_models", "zip", "artifacts/models")
!ls -la artifacts/models